<a href="https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/integration/mempalace_with_milvus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>   <a href="https://github.com/milvus-io/bootcamp/blob/master/integration/mempalace_with_milvus.ipynb" target="_blank">    <img src="https://img.shields.io/badge/View%20on%20GitHub-555555?style=flat&logo=github&logoColor=white" alt="GitHub Repository"/></a># MemPalace with Milvus[MemPalace](https://github.com/MemPalace/mempalace) is a memory layer for coding agents and long-running development workflows. It can mine and search project memories, conversation notes, and other "drawers" that help an agent keep useful context across sessions.In this tutorial, we will configure MemPalace to use [Milvus](https://milvus.io/) as its storage backend. The notebook uses Milvus Lite by default, so it can run locally or in Google Colab without a separate Milvus server. The same MemPalace configuration can also point to Milvus server or Zilliz Cloud for shared or production deployments.## PrerequisitesInstall MemPalace with the Milvus backend dependencies from GitHub.

In [ ]:
! pip install --upgrade "mempalace[milvus] @ git+https://github.com/MemPalace/mempalace.git@develop"

> If you are using Google Colab, to enable dependencies just installed, you may need to **restart the runtime** (click on the "Runtime" menu at the top of the screen, and select "Restart session" from the dropdown menu).This tutorial uses MemPalace's local embedding model, so you do not need an external model API key. The first run may download a small ONNX embedding model.## Configure MemPalace to use MilvusMemPalace can work with a real project directory through its CLI, but a notebook is easier to run when it creates a temporary palace directory and inserts a small set of example memories. We set the backend to `milvus`, force CPU embeddings, and keep the demo inside a temporary folder.

In [ ]:
import os
import tempfile
from pathlib import Path

os.environ["MEMPALACE_BACKEND"] = "milvus"
os.environ["MEMPALACE_EMBEDDING_MODEL"] = "minilm"
os.environ["MEMPALACE_EMBEDDING_DEVICE"] = "cpu"
os.environ["MEMPALACE_EMBEDDING_THREADS"] = "2"

work_dir = Path(tempfile.mkdtemp(prefix="mempalace_milvus_demo_"))
palace_dir = work_dir / "palace"
palace_dir.mkdir(parents=True, exist_ok=True)

print(f"Palace directory: {palace_dir}")

When no remote Milvus URI is configured, the MemPalace Milvus backend creates a local Milvus Lite database at `<palace>/milvus.db`.> As for the argument of `MilvusClient` used by the backend:> - Setting the `uri` as a local file, e.g. `./milvus.db`, is the most convenient method, as it automatically utilizes [Milvus Lite](https://milvus.io/docs/milvus_lite.md) to store all data in this file.> - If you have large scale of data, you can set up a more performant Milvus server on [Docker or Kubernetes](https://milvus.io/docs/quickstart.md). In this setup, please use the server uri, e.g. `http://localhost:19530`, as your `uri`.> - If you want to use [Zilliz Cloud](https://zilliz.com/cloud), the fully managed cloud service for Milvus, adjust the `uri` and `token`, which correspond to the [Public Endpoint and API key](https://docs.zilliz.com/docs/on-zilliz-cloud-console#free-cluster-details) in Zilliz Cloud.## Create a MemPalace collectionMemPalace exposes `get_collection` as the main Python API for opening a memory drawer collection. Passing `backend="milvus"` makes the collection use Milvus while still letting us call MemPalace methods such as `upsert`, `query`, and `lexical_search`.

In [ ]:
from mempalace.palace import get_collection

collection_name = "mempalace_drawers"
collection = get_collection(
    str(palace_dir),
    collection_name,
    create=True,
    backend="milvus",
)

print(type(collection).__name__)

## Add example project memoriesMemPalace organizes memory as `wings`, `rooms`, and `drawers`: a wing usually represents a project or person, a room represents a topic area, and a drawer stores the original memory text. Higher-level MemPalace flows such as `mempalace mine` can create these drawers from a project directory or conversation export. In this notebook, we write a few drawers directly so the Milvus integration is easy to see.The records below mimic a team using MemPalace as long-term project memory for a coding agent. MemPalace computes embeddings locally and stores the text, metadata, dense vectors, and BM25 lexical index in Milvus.

In [ ]:
memories = [
    {
        "id": "mem-001",
        "text": (
            "The team uses MemPalace as a long-term memory layer for coding agents, "
            "with Milvus storing searchable drawers for architecture decisions and runbooks."
        ),
        "metadata": {
            "wing": "milvus_memory_assistant",
            "room": "architecture",
            "source_file": "architecture-notes.md",
        },
    },
    {
        "id": "mem-002",
        "text": (
            "Local development uses Milvus Lite so engineers can run the memory stack "
            "without a separate vector database service."
        ),
        "metadata": {
            "wing": "milvus_memory_assistant",
            "room": "deployment",
            "source_file": "local-dev-runbook.md",
        },
    },
    {
        "id": "mem-003",
        "text": (
            "For shared environments, the same MemPalace backend can point to Milvus "
            "server or Zilliz Cloud by setting MEMPALACE_MILVUS_URI and MEMPALACE_MILVUS_TOKEN."
        ),
        "metadata": {
            "wing": "milvus_memory_assistant",
            "room": "deployment",
            "source_file": "cloud-deployment.md",
        },
    },
    {
        "id": "mem-004",
        "text": (
            "The backend keeps verbatim memory text and metadata, while Milvus stores "
            "vectors and supports semantic search over drawers."
        ),
        "metadata": {
            "wing": "milvus_memory_assistant",
            "room": "architecture",
            "source_file": "storage-contract.md",
        },
    },
    {
        "id": "mem-005",
        "text": (
            "The evaluation checklist asks the agent to verify semantic search, "
            "metadata filters, and lexical search before publishing an integration guide."
        ),
        "metadata": {
            "wing": "milvus_memory_assistant",
            "room": "evaluation",
            "source_file": "evaluation-checklist.md",
        },
    },
    {
        "id": "mem-006",
        "text": (
            "When a query includes exact terms such as Zilliz Cloud token, the agent "
            "should use lexical search as a complement to vector similarity."
        ),
        "metadata": {
            "wing": "milvus_memory_assistant",
            "room": "retrieval",
            "source_file": "retrieval-playbook.md",
        },
    },
]

collection.upsert(
    ids=[memory["id"] for memory in memories],
    documents=[memory["text"] for memory in memories],
    metadatas=[memory["metadata"] for memory in memories],
)

print(f"Inserted {collection.count()} memories")

## Semantic searchUse `query_texts` to search memories by meaning. MemPalace embeds the query text with the same local embedding model and sends the vector search to Milvus.

In [ ]:
semantic_results = collection.query(
    query_texts=["How can I run MemPalace memory locally without a database server?"],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

for rank, (item_id, document, metadata, distance) in enumerate(
    zip(
        semantic_results.ids[0],
        semantic_results.documents[0],
        semantic_results.metadatas[0],
        semantic_results.distances[0],
    ),
    start=1,
):
    print(f"{rank}. {item_id} distance={distance:.4f} room={metadata.get('room')}")
    print(f"   {document}")

## Search within a metadata scopeMetadata filters are useful when a project has many memory rooms. This query searches only deployment-related memories.

In [ ]:
filtered_results = collection.query(
    query_texts=["How do we deploy this memory backend?"],
    n_results=3,
    where={"room": "deployment"},
    include=["documents", "metadatas", "distances"],
)

for rank, (item_id, document, metadata, distance) in enumerate(
    zip(
        filtered_results.ids[0],
        filtered_results.documents[0],
        filtered_results.metadatas[0],
        filtered_results.distances[0],
    ),
    start=1,
):
    print(f"{rank}. {item_id} distance={distance:.4f} source={metadata.get('source_file')}")
    print(f"   {document}")

## Lexical searchThe Milvus backend also creates a BM25 sparse index over the memory text. `lexical_search` is useful when you need exact terms such as environment variable names, product names, or file names.

In [ ]:
lexical_hits = collection.lexical_search(query="Zilliz Cloud token", n_results=3).hits

for rank, hit in enumerate(lexical_hits, start=1):
    print(f"{rank}. {hit.id} score={hit.score:.4f} room={hit.metadata.get('room')}")
    print(f"   {hit.document}")

## Inspect the Milvus collectionMemPalace manages the Milvus schema for us. To confirm what was created, we can open the same Milvus Lite file with `MilvusClient` and inspect the collection directly.

In [ ]:
from pymilvus import MilvusClient

collection.close()

milvus_uri = str(palace_dir / "milvus.db")
client = MilvusClient(uri=milvus_uri)

print(client.list_collections())
print(client.get_collection_stats(collection_name))

rows = client.query(
    collection_name=collection_name,
    filter="",
    limit=3,
    output_fields=["id", "document", "metadata"],
)

rows

## Optional: use Milvus server or Zilliz CloudFor a shared deployment, set the Milvus connection environment variables before opening the MemPalace collection. Leave them unset to use the local Milvus Lite file shown above.

In [ ]:
# os.environ["MEMPALACE_MILVUS_URI"] = "https://your-cluster.api.region.zillizcloud.com"
# os.environ["MEMPALACE_MILVUS_TOKEN"] = "***********"
# os.environ["MEMPALACE_MILVUS_DB_NAME"] = "default"
# os.environ["MEMPALACE_MILVUS_NAMESPACE"] = "team-memory"

For a real project, you usually do not need to hand-write the drawers as we did in the small Python example above. Use the MemPalace CLI to mine project files or conversation exports, and pass `--backend milvus` so those generated drawers are stored in Milvus:```shellmempalace init /path/to/project --backend milvus --yesmempalace mine /path/to/project --backend milvusmempalace search "How do we deploy memory?" --backend milvus```## ConclusionMemPalace gives agents a structured way to preserve project context: wings keep domains separate, rooms make memory scoping explicit, and drawers keep the original text available for retrieval. Milvus provides the storage and search layer behind that structure, combining vector search, metadata filtering, and lexical retrieval in one backend. Starting with Milvus Lite keeps the notebook simple, while the same integration path can scale to Milvus server or Zilliz Cloud when the memory layer needs to serve a team or a production agent workflow.